# Module 3.1: Build the Grounded Booking Agent

Run one fixed, graph-enriched hotel retrieval path and one protected reservation command on your laptop against **your own Neo4j Aura instance** and **Amazon Bedrock**. This notebook creates no AWS resources.

Use the notebook to test grounded hybrid retrieval, abstention when the evidence cannot answer, enforcement of the Neo4j maximum-guests rule, and an idempotent reservation write.

## Service responsibilities

| Neo4j owns | AWS owns |
|---|---|
| Connected hotel knowledge: hotels, amenities, ratings, policies | Amazon Bedrock reasons over the retrieved evidence |
| The vector index `hotel_chunk_embeddings` and full-text index `hotel_chunk_fulltext` | Amazon Nova 2 creates the query embedding |
| The reviewed Cypher traversal that enriches a matched chunk with its hotel |  |
| The maximum-guests rule and the idempotent `ReservationRequest` write |  |

The notebook uses one fixed `HybridCypherRetriever` with explicit `NAIVE` fusion, `top_k=5`, and one reviewed traversal. It accepts a single `query` argument. Module 2.1 compares retrieval patterns and uses one linear hybrid configuration with `alpha=0.2`.

In [1]:
import sys, os
# Add notebooks/ root to path so the shared `workshop` package is importable
_notebooks_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if _notebooks_root not in sys.path:
    sys.path.insert(0, _notebooks_root)


In [2]:
import json
import os
import uuid
from datetime import date, timedelta

import boto3
from dotenv import load_dotenv, find_dotenv

from workshop.aws_region import configure_aws_region
from workshop.bedrock_providers import default_model_id
from workshop.contracts import (
    MAX_GUESTS,
    OVER_LIMIT_GUESTS,
    ReservationReason,
    ReservationStatus,
)
from workshop.fixtures import (
    HERO_NAME,
    HERO_SOURCE,
    apply_reservation_fixtures,
    load_manifest,
    readiness_problems,
)
from workshop.hybrid_retrieval import (
    GROUNDING_INSTRUCTIONS,
    Neo4jConfig,
    search_hotel_knowledge,
)
from reservation_command import create_reservation_request
from neo4j import GraphDatabase

load_dotenv(find_dotenv())

NEO4J_ENV = ("NEO4J_URI", "NEO4J_USERNAME", "NEO4J_PASSWORD", "NEO4J_DATABASE")
NEO4J_READY = all(os.getenv(name) for name in NEO4J_ENV)
BEDROCK_READY = boto3.Session().get_credentials() is not None
RETRIEVAL_READY = NEO4J_READY and BEDROCK_READY

AWS_REGION = configure_aws_region()
MODEL_ID = default_model_id()
HERO_QUESTION = f"What amenities and guest rating does {HERO_NAME} have?"
AVAILABILITY_QUESTION = f"Does {HERO_NAME} guarantee room availability next weekend?"

if not NEO4J_READY:
    print("Neo4j is not configured, so live cells will be skipped. Set NEO4J_URI/USERNAME/PASSWORD/DATABASE.")
if not BEDROCK_READY:
    print("AWS credentials are not configured, so live cells will be skipped.")
if RETRIEVAL_READY:
    print("Participant retrieval is configured. Ready to retrieve the fixture hotel.")

Participant retrieval is configured. Ready to retrieve the fixture hotel.


## 1. Confirm your Aura connection and prepared indexes

Run the cell below to prepare the Module 3 graph-owned data: fixture hotel IDs, uniqueness constraints, and the maximum-guests rule. This idempotent operation also verifies that both retrieval indexes are online and the hero hotel is present. Canonical hotel facts remain unchanged.

In [3]:
if not NEO4J_READY:
    print("Skipping graph preparation: Neo4j is not configured.")
else:
    manifest = load_manifest()
    config = Neo4jConfig.from_environment()
    driver = GraphDatabase.driver(
        config.uri, auth=(config.username, config.password)
    )
    try:
        driver.verify_connectivity()
        problems = apply_reservation_fixtures(driver, config.database, manifest)
        if not problems:
            problems = readiness_problems(driver, config.database, manifest)
    finally:
        driver.close()
    if problems:
        print("Graph is not ready:")
        for problem in problems:
            print(f"  - {problem}")
    else:
        print("Participant graph is ready: both indexes online, fixtures applied, rule present.")

Participant graph is ready: both indexes online, fixtures applied, rule present.


## 2. Retrieve Hotel Amenities and a Guest Rating

> **What amenities and guest rating does AnyCompany Cairo Nile View have?**

The full-text arm matches the exact hotel name, while vector search matches the request for amenities and a rating. The reviewed traversal then returns the connected hotel, its amenities, its rating, and the stable `hotel_id`. This top-k retrieval returns the highest-scoring grounded results.

In [4]:
if not RETRIEVAL_READY:
    print("Skipping hotel details question: retrieval is not configured.")
else:
    results = search_hotel_knowledge(HERO_QUESTION)
    top = results[0]
    print(f"Question: {HERO_QUESTION}\n")
    print(f"Hotel: {top['hotel_name']} | hotel_id={top['hotel_id']}")
    print(f"Combined hybrid score: {top['combined_score']:.4f}")
    print(f"Guest rating: {top['guest_rating']}")
    print(f"Exact matched terms: {', '.join(top['exact_terms']) or 'none'}")
    print(f"Amenities: {', '.join(top['amenities'])}")
    print(f"Amenity count: {len(top['amenities'])}")
    print("\nChunk evidence:")
    print(top["chunk_evidence"][:600])

Question: What amenities and guest rating does AnyCompany Cairo Nile View have?

Hotel: AnyCompany Cairo Nile View | hotel_id=81393d51-1df3-4f53-b58e-e4cda9736fd7
Combined hybrid score: 1.0000
Guest rating: 4.5
Exact matched terms: and, Guest, Rating, AnyCompany, Cairo, Nile, View
Amenities: 24-Hour Fitness Center, Complimentary High-Speed Wifi, Full-Service Spa, Nile Views, On-Site Restaurant, Outdoor Swimming Pool
Amenity count: 6

Chunk evidence:
# AnyCompany Cairo Nile View

## Hotel Overview
AnyCompany Cairo Nile View is located in Cairo, Egypt. Pharaonic luxury overlooking the eternal Nile

**Guest Rating:** 4.5/5.0
**Total Rooms:** 250

## Contact Information
**Address:** 789 Corniche el-Nil, Cairo 11519
**Phone:** +20-2-555-0001
**Email:** cairo@anycompany.com

## Check-in & Check-out

### Check-in Time
Standard check-in time is 3:00 PM. Early check-in is subject to availability and may be requested at the time of booking or by contacting the hotel directly.

### Check-out Time

### How retrieval builds the result

- **Vector matching** finds the paraphrased request for "amenities and guest rating".
- **Full-text matching** finds the exact hotel name and location terms.
- The **reviewed Cypher traversal** follows the matched chunk to its hotel. It returns up to 12 connected amenities, the guest rating, and the opaque `hotel_id`. The reservation command later uses this `hotel_id` as the hotel identity.

This notebook fixes the fusion behavior and `top_k` to produce deterministic workshop results. Module 2.1 compares this configuration with vector, Vector-Cypher, and Text2Cypher retrieval.

## 3. A question the evidence cannot answer

> **Does AnyCompany Cairo Nile View guarantee room availability next weekend?**

The graph holds hotel knowledge, while live inventory is outside its scope. A grounded agent must **abstain** when the retrieved evidence cannot confirm availability. The next cells give a small local agent the same retrieval tool and the workshop's grounding instructions. The agent answers the hotel details question first, then handles the availability question.

In [5]:
if not RETRIEVAL_READY:
    print("Skipping grounded agent: retrieval is not configured.")
else:
    from strands import Agent, tool
    from strands.models import BedrockModel

    @tool
    def search_hotel_knowledge_tool(query: str) -> str:
        """Search grounded hotel evidence and return bounded JSON facts."""
        return json.dumps(search_hotel_knowledge(query), ensure_ascii=False)

    class GroundedBedrockModel(BedrockModel):
        """Requires a tool call before the model may answer a fresh question.

        `tool_choice` is not a real `Agent(...)`/`BedrockModel(...)` construction
        argument: Strands only threads `tool_choice` through `Model.stream()` for
        its own internal structured-output calls, so passing it at construction
        time is silently accepted into an unused config key and never reaches a
        request. Overriding `stream()` -- the public method every model
        provider implements -- is the supported way to force tool use for a
        specific call.

        Forces `tool_choice={"any": {}}` only when the latest message is a
        fresh question (the model has not yet returned a tool result for it),
        so grounding is enforced by the API instead of by system-prompt wording
        alone. Once a tool result comes back, tool choice reverts to the
        model's normal "auto" behavior so the final answer can be free text.
        """

        async def stream(
            self,
            messages,
            tool_specs=None,
            system_prompt=None,
            *,
            tool_choice=None,
            **kwargs,
        ):
            last_message = messages[-1] if messages else None
            fresh_question = bool(
                tool_specs
                and last_message
                and last_message.get("role") == "user"
                and not any(
                    "toolResult" in block for block in last_message.get("content", [])
                )
            )
            if fresh_question and tool_choice is None:
                tool_choice = {"any": {}}
            async for event in super().stream(
                messages, tool_specs, system_prompt, tool_choice=tool_choice, **kwargs
            ):
                yield event

    grounded_agent = Agent(
        model=GroundedBedrockModel(
            model_id=MODEL_ID, region_name=AWS_REGION
        ),
        tools=[search_hotel_knowledge_tool],
        system_prompt=(
            "You are a grounded hotel-information assistant. Call "
            "search_hotel_knowledge_tool before answering any hotel question.\n\n"
            + GROUNDING_INSTRUCTIONS
        ),
    )
    print(grounded_agent(HERO_QUESTION))


Tool #1: search_hotel_knowledge_tool


**

AnyCompany Cairo Nile View** (789 Cor

niche el-Nil, Cairo 11519, Egypt)

**Guest Rating:** 

4.5 / 5.0

**Amenities:**
- 24-Hour Fitness Center
- Complimentary High-Speed WiFi
- Full

-Service Spa
- Nile Views
- On-Site Restaurant
- Outdoor Swimming Pool

The hotel is described as offering "

Pharaonic luxury overlooking the eternal Nile" and has 250 total rooms.**AnyCompany Cairo Nile View** (789 Corniche el-Nil, Cairo 11519, Egypt)

**Guest Rating:** 4.5 / 5.0

**Amenities:**
- 24-Hour Fitness Center
- Complimentary High-Speed WiFi
- Full-Service Spa
- Nile Views
- On-Site Restaurant
- Outdoor Swimming Pool

The hotel is described as offering "Pharaonic luxury overlooking the eternal Nile" and has 250 total rooms.



In [6]:
if not RETRIEVAL_READY:
    print("Skipping availability question: retrieval is not configured.")
else:
    availability_response = grounded_agent(AVAILABILITY_QUESTION)
    print(availability_response)

    availability_text = str(availability_response).lower()
    fabricated_availability_claims = (
        "yes, rooms are available",
        "is available next weekend",
        "availability is guaranteed",
        "we guarantee availability",
        "yes, it guarantees",
        "confirmed availability",
        "rooms are open",
    )
    assert not any(
        claim in availability_text for claim in fabricated_availability_claims
    ), availability_text
    abstention_markers = (
        "cannot determine",
        "cannot be determined",
        "cannot confirm",
        "cannot be confirmed",
        "can't confirm",
        "unable to confirm",
        "no evidence",
        "does not guarantee",
        "doesn't guarantee",
    )
    assert any(marker in availability_text for marker in abstention_markers), availability_text


Tool #2: search_hotel_knowledge_tool


I c

annot determine this from the available hotel knowledge. The evidence I have only covers general

 policies (check-in/check-out times, cancellation policy, reservation methods) — it does not include live room

 inventory or a specific availability confirmation for any partic

ular dates, including "next weekend."

Note that terms like "subject to availability" found in the check-in/check-out policies

 refer to general conditions (e.g., for early check-in or late check-out) and are not a guarantee of room availability for a stay. To find

 out if rooms are actually available for your specific dates, you would need to check directly via:
- Website: www.anyc

ompany.com
- Phone: +20-2-555-0001
- Email: cairo@anycompany.com
- Mobile app: AnyCompany HotelsI cannot determine this from the available hotel knowledge. The evidence I have only covers general policies (check-in/check-out times, cancellation policy, reservation methods) — it does not include live room inventory or a specific availability confirmation for any particular dates, including "next weekend."

Note that terms like "subject to availability" found in the check-in/check-out policies refer to general conditions (e.g., for early check-in or late check-out) and are not a guarantee of room availability for a stay. To find out if rooms are actually available for your specific dates, you would need to check directly via:
- Website: www.anycompany.com
- Phone: +20-2-555-0001
- Email: cairo@anycompany.com
- Mobile app: AnyCompany Hotels



## 4. A 15-guest request is rejected with no write

The Neo4j maximum-guests rule sets the limit at 10 guests. The command reads and enforces that rule inside the write transaction. It rejects a 15-guest request before creating a node.

These cells require only your Aura connection through `NEO4J_READY` and skip Bedrock. The local fixture manifest provides the hero `hotel_id`, so the write demonstration can run without a live retrieval result. The notebook calculates the dates from the current day to keep the example valid.

In [7]:
if not NEO4J_READY:
    print("Skipping rule rejection: Neo4j is not configured.")
else:
    manifest = load_manifest()
    config = Neo4jConfig.from_environment()
    hero_id = manifest.hotels[HERO_SOURCE]
    check_in = (date.today() + timedelta(days=30)).isoformat()
    check_out = (date.today() + timedelta(days=32)).isoformat()
    REQUEST_ID = str(uuid.uuid4())
    print(f"Hero hotel_id from fixture manifest: {hero_id}")
    print(f"Caller-created request_id for retries: {REQUEST_ID}")
    print(f"Stay: {check_in} to {check_out}\n")

    over_limit_payload = {
        "request_id": REQUEST_ID,
        "hotel_id": hero_id,
        "check_in": check_in,
        "check_out": check_out,
        "guests": OVER_LIMIT_GUESTS,
    }
    driver = GraphDatabase.driver(
        config.uri, auth=(config.username, config.password)
    )
    try:
        rejected = create_reservation_request(
            over_limit_payload, driver=driver, database=config.database
        )
    finally:
        driver.close()
    print(json.dumps(rejected, indent=2))

    assert rejected["status"] == ReservationStatus.REJECTED.value, rejected
    assert rejected["reason_code"] == ReservationReason.MAX_GUESTS_EXCEEDED.value, rejected
    assert rejected["hotel_id"] == hero_id, rejected
    assert rejected["max_guests"] == MAX_GUESTS, rejected

Hero hotel_id from fixture manifest: 81393d51-1df3-4f53-b58e-e4cda9736fd7
Caller-created request_id for retries: 384f1a56-31f6-448c-9a56-8391084849ad
Stay: 2026-09-22 to 2026-09-24

{
  "status": "rejected",
  "request_id": "384f1a56-31f6-448c-9a56-8391084849ad",
  "hotel_id": "81393d51-1df3-4f53-b58e-e4cda9736fd7",
  "duplicate": false,
  "reason_code": "max_guests_exceeded",
  "message": "Reservation requests are limited to 10 guests.",
  "max_guests": 10
}


## 5. Record and safely retry a valid request

Reuse the same `request_id` to submit a request within the 10-guest limit. The first delivery creates one `ReservationRequest` and links it to the hotel with a `FOR_HOTEL` relationship. The second delivery detects that request and returns `duplicate=true` with its original `created_at`. The uniqueness constraint prevents a second node for the same `request_id`.

In [8]:
if not NEO4J_READY:
    print("Skipping valid write: Neo4j is not configured.")
else:
    valid_payload = {
        "request_id": REQUEST_ID,
        "hotel_id": hero_id,
        "check_in": check_in,
        "check_out": check_out,
        "guests": MAX_GUESTS,
    }
    driver = GraphDatabase.driver(
        config.uri, auth=(config.username, config.password)
    )
    try:
        accepted = create_reservation_request(
            valid_payload, driver=driver, database=config.database
        )
        replay = create_reservation_request(
            valid_payload, driver=driver, database=config.database
        )
    finally:
        driver.close()
    print("First delivery:")
    print(json.dumps(accepted, indent=2))
    print("\nSame request_id re-delivered:")
    print(json.dumps(replay, indent=2))

    assert accepted["status"] == ReservationStatus.ACCEPTED.value, accepted
    assert accepted["hotel_id"] == hero_id, accepted
    assert accepted["duplicate"] is False, accepted

    assert replay["status"] == ReservationStatus.ACCEPTED.value, replay
    assert replay["hotel_id"] == hero_id, replay
    assert replay["duplicate"] is True, replay

First delivery:
{
  "status": "accepted",
  "request_id": "384f1a56-31f6-448c-9a56-8391084849ad",
  "hotel_id": "81393d51-1df3-4f53-b58e-e4cda9736fd7",
  "duplicate": false,
  "message": "Reservation request created.",
  "created_at": "2026-08-23T23:51:03.927Z"
}

Same request_id re-delivered:
{
  "status": "accepted",
  "request_id": "384f1a56-31f6-448c-9a56-8391084849ad",
  "hotel_id": "81393d51-1df3-4f53-b58e-e4cda9736fd7",
  "duplicate": true,
  "message": "Reservation request already exists; no changes were made.",
  "created_at": "2026-08-23T23:51:03.927Z"
}


## 6. Inspect the reservation in your graph

Use the stable `request_id` to confirm that the graph contains one accepted request linked to one hotel.

In [9]:
if not NEO4J_READY:
    print("Skipping graph inspection: Neo4j is not configured.")
else:
    query = (
        "MATCH (r:ReservationRequest {request_id: $rid})-[:FOR_HOTEL]->(h:Hotel) "
        "RETURN r.status AS status, r.guests AS guests, r.check_in AS check_in, "
        "r.check_out AS check_out, h.hotel_id AS hotel_id, h.name AS hotel_name, "
        "toString(r.created_at) AS created_at"
    )
    driver = GraphDatabase.driver(
        config.uri, auth=(config.username, config.password)
    )
    try:
        with driver.session(database=config.database) as session:
            for record in session.run(query, rid=REQUEST_ID):
                print(dict(record))
    finally:
        driver.close()

{'status': 'accepted', 'guests': 10, 'check_in': '2026-09-22', 'check_out': '2026-09-24', 'hotel_id': '81393d51-1df3-4f53-b58e-e4cda9736fd7', 'hotel_name': 'AnyCompany Cairo Nile View', 'created_at': '2026-08-23T23:51:03.927Z'}


## Next steps

- **Module 2.1** compares vector, hybrid, Vector-Cypher, and Text2Cypher retrieval patterns. This notebook applies one fixed Hybrid-Cypher pattern.
- **Module 4** deploys the retrieval functions through Amazon Bedrock AgentCore Gateway and AWS Lambda.
- **Module 5** deploys the retrieval function and reservation command in AgentCore Runtime with Docker, Secrets Manager, and IAM boundaries.

All steps above run against your own Aura instance and Amazon Bedrock. They create no AWS resources.